# Phase 1 Demo — Land Cover Classification

Final Master's Project (Master's in Artificial Intelligence and Data Analysis, FP-UNA).

This notebook demonstrates **Phase 1 in full**: the Sentinel-2 + MapBiomas dataset, the Random
Forest vs. CNN comparison, and the model-selection criterion. Every hyperparameter cell carries
its own explanation — meant to let you change a value live and explain what it does.

Companion references in the repo: `docs/RECORRIDO_PASO_A_PASO.md` (full narrative walkthrough,
Spanish) and `docs/METODOLOGIA_PIPELINE.md` (technical reference).

## Setup (one-time)

This notebook runs on the dataset that's already been built (1,193 patches of 33×33 pixels) — it
doesn't pull anything from Google Earth Engine again, so the demo is fast and doesn't depend on
the professor having access to a GEE project.

**Before the demo, once:**
1. Upload `notebooks/phase1_demo_bundle.zip` (generated from the repo) to your Google Drive, at
   `MyDrive/phase1_demo/phase1_demo_bundle.zip`.
2. Run the cell below — it mounts Drive and unpacks the dataset + shared modules
   (`dataset_loader.py`, `classifier_report.py`, the exact same ones the real pipeline uses, not
   a rewritten copy) into `/content/marta/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
from pathlib import Path

BUNDLE_ZIP = "/content/drive/MyDrive/phase1_demo/phase1_demo_bundle.zip"
CONTENT_ROOT = Path("/content")
REPO_ROOT = CONTENT_ROOT / "marta"  # matches the repo's actual directory name on disk

if not REPO_ROOT.exists():
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        z.extractall(CONTENT_ROOT)

n_patches = sum(1 for _ in (REPO_ROOT / "data/study_area/dataset").rglob("*.npy"))
print(f"Patches found: {n_patches}")


In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import dataset_loader as dl
import classifier_report

print("Classes:", dl.CLASS_NAMES)


---
## Step 0 — Why this project exists

In the Paraguayan Chaco, some projects sell "carbon credits": they promise a company that paying
them prevents deforestation of a tract of land, which in turn prevents CO2 from being released
into the atmosphere. To sell those credits, the project has to show an auditor how much forest it
has and how much carbon it stores.

The real problem **isn't the absence of automation** — satellite classification tools that run at
scale already exist (Global Forest Watch, MapBiomas, used officially by MADES). The problem is
that none of them give the auditor **traceable evidence per individual prediction**: they get a
final number they have to trust, not evidence they can review case by case. A study published in
*Science* (2025) found that, on average, REDD+ projects worldwide issue 10.7 times more credits
than are actually justified.

**Research question**: can a system be built, using free satellite imagery, that classifies land
cover with interpretable evidence for each prediction, and translates the detected change into
tons of CO2 — in an auditable way? This notebook covers the first half (classification); carbon
estimation is Phase 2, not yet implemented.

## Step 1 — What imagery we use, and why

- **Imagery: Sentinel-2 (Copernicus/ESA), L2A level** — 10m resolution, 13 spectral bands (not
  just red/green/blue: it includes near-infrared and red-edge, where healthy vegetation is
  distinguished from stressed vegetation or bare soil more clearly). L2A (surface reflectance,
  atmospherically corrected) rather than L1C (top of atmosphere) because the 2019 vs. 2023
  temporal comparison needs reflectance that's comparable across dates — with L1C, different
  atmospheric conditions on each date would introduce differences that aren't real land-cover
  change.
- **Labels: MapBiomas Chaco, Collection 5** — an already-existing classification, 1985-2023,
  specific to the Gran Chaco. Downside: 30m resolution (derived from Landsat), 3x coarser than a
  Sentinel-2 pixel (picked back up in Step 3).

In [ ]:
from collections import Counter

for split in ["train", "val", "test"]:
    rows = dl.load_manifest(split)
    counts = Counter(r["class"] for r in rows)
    print(f"{split:5s} ({len(rows):4d} patches):", {k: counts.get(k, 0) for k in dl.CLASS_NAMES})


## Step 2 — The 3 study areas

1. **Case-study AOI**: Corazón Verde del Chaco (VCS 2611), ~20,589 ha — the real tract of land
   that would be audited in Phase 4. It doesn't have a single "cultivo" (cropland) pixel in
   either year.
2. **Sampling footprint**: the 3 Paraguayan Chaco departments (Alto Paraguay, Boquerón,
   Presidente Hayes), 24 million hectares — this is where the 1,193 training patches come from,
   since the AOI alone has no cropland examples. Deterministic sampling (`seed=42`), not random
   across runs.
3. **Held-out sub-areas, excluded from all training**: 3 independent regions used to check
   generalization — Filadelfia (Boquerón, cross-checked against ESA WorldCover, a source never
   used for training, unlike MapBiomas), Bahía Negra, and Pozo Colorado. *These checks already
   ran (`phase1-classifier` task 5, 5 seeds each): the CNN wins or ties in 2 of the 3 (Pozo
   Colorado, and an effective tie in Bahía Negra), and loses only in Filadelfia, by a margin wider
   than its own win on the AOI's test split below. Picked back up in Step 5.*

### What a patch looks like, by class

True color (bands B4/B3/B2 = red/green/blue), surface reflectance scaled to [0, 1] for
visualization only — the model trains on raw values, without this normalization.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

RGB_IDX = [3, 2, 1]  # B4, B3, B2 in S2_BANDS (see scripts/s2_utils.py)
N_EXAMPLES = 4

fig, axes = plt.subplots(len(dl.CLASS_NAMES), N_EXAMPLES, figsize=(2.2 * N_EXAMPLES, 2.2 * len(dl.CLASS_NAMES)))
for row, class_name in enumerate(dl.CLASS_NAMES):
    examples = [r for r in dl.load_manifest("train") if r["class"] == class_name][:N_EXAMPLES]
    for col, r in enumerate(examples):
        patch = np.load(REPO_ROOT / r["path"])
        rgb = np.clip(patch[:, :, RGB_IDX] / 3000, 0, 1)
        ax = axes[row, col]
        ax.imshow(rgb)
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(class_name, fontsize=12)
fig.suptitle("Example patches per class (true color)")
fig.tight_layout()
plt.show()


## Step 3 — How a patch is built, and the spatial leak that nearly invalidated the result

MapBiomas (30m) and Sentinel-2 (10m) have an exact 3:1 ratio — each MapBiomas pixel is a 3x3
block of Sentinel-2 pixels, the **label alignment unit**. The final patch is **33x33** (not 3x3):
a 9-pixel input barely allows a meaningful convolution, and it would defeat the point of the
CNN-vs-RF comparison below — the CNN needs real spatial context to have any chance of beating a
per-pixel classifier.

**The data leak (the most important part of this section)**: each patch is 330m on a side — two
patches less than that real distance apart can overlap on the ground. Splitting points randomly
across train/val/test can send two patches from the same patch of land to opposite sides of the
split (a leak), artificially inflating the metric. With a leaky version, the conclusion had been
"RF wins clearly" (0.826 vs. 0.770 macro F1); once fixed, it changed to "a technical tie" — the
cell below verifies live that the fix holds.

In [ ]:
from scipy.spatial import cKDTree

def to_meters_xy(lon, lat, lat0):
    """Simple equirectangular projection, good enough for short distances within the Chaco."""
    R = 6371000
    x = np.radians(lon) * R * np.cos(np.radians(lat0))
    y = np.radians(lat) * R
    return x, y

all_rows = dl.load_manifest()
lat0 = np.mean([float(r["lat"]) for r in all_rows])
coords = np.array([to_meters_xy(float(r["lon"]), float(r["lat"]), lat0) for r in all_rows])
splits = np.array([r["split"] for r in all_rows])

tree = cKDTree(coords)
PATCH_SIZE_M = 330
min_cross_split_dist = np.inf
for i, (pt, split) in enumerate(zip(coords, splits)):
    idx = tree.query_ball_point(pt, r=PATCH_SIZE_M * 6)
    for j in idx:
        if splits[j] != split and j != i:
            d = np.linalg.norm(coords[i] - coords[j])
            min_cross_split_dist = min(min_cross_split_dist, d)

print(f"Closest pair across different splits: {min_cross_split_dist:.0f} m")
print(f"Leakage threshold (patch size): {PATCH_SIZE_M} m")
print("No leakage" if min_cross_split_dist >= PATCH_SIZE_M else "LEAKAGE DETECTED")


## Step 4 — The two models, and how they work inside

**Random Forest**: a chain of decision trees, each one asking about band values of a single
pixel ("is B11 > threshold?"), averaging the vote of hundreds of trees trained on different
bootstrap samples and band subsets. It has no mechanism that knows a pixel is "next to" another
one — it only ever looks at spectral values.

**CNN**: slides small filters (3x3) over the whole patch, building increasingly abstract patterns
layer by layer — unlike RF, it can actually exploit how neighboring pixels relate to each other.
That's the capability the experiment below puts to the test.

### Random Forest — hyperparameters

The two that can be tweaked live are in the training cell below, as constants at the top (same
as in `scripts/09_train_random_forest.py`):

- **`N_ESTIMATORS` (400)** — number of trees. Each individual tree overfits its own bootstrap
  sample; averaging hundreds of trees that get it wrong in different ways cancels out most of
  that error. Typical range 300-500 — going higher usually doesn't help (diminishing returns),
  going much lower (e.g. 20) makes the average less stable and more sensitive to the seed.
- **`max_depth=None`** — maximum depth of each tree, unlimited. In a *single* tree this would be
  dangerous (it memorizes its own data's noise); in a Random Forest it isn't, because the
  overfitting reduction comes from averaging trees that differ from each other, not from pruning
  each one. Limiting it (e.g. to 10) trains faster and uses less memory, at some accuracy cost —
  not needed at this data scale.
- **`random_state` (the seed)** — controls which bootstrap sample and which band subset each tree
  sees. Changing it shouldn't move the result much if the model is stable — which is exactly what
  the 5-seed comparison in Step 5 puts to the test.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

N_ESTIMATORS = 400   # change this for the live demo
MAX_DEPTH = None     # change this for the live demo
SEED = 42

X_train, y_train = dl.load_pixel_features("train")
print("Class distribution in train:", Counter(dl.CLASS_NAMES[c] for c in y_train))

rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
print("Random Forest trained.")


In [ ]:
X_test, y_test = dl.load_pixel_features("test")
rf_pred = rf.predict(X_test)
rf_results = classifier_report.evaluate_predictions(y_test, rf_pred)
classifier_report.print_evaluation(rf_results, "Random Forest")

print("\nBand importance (B1..B12, NDVI):")
for name, imp in zip(["B1","B2","B3","B4","B5","B6","B7","B8","B8A","B9","B11","B12","NDVI"], rf.feature_importances_.round(3)):
    print(f"  {name}: {imp}")


### CNN — architecture, and why it's small and homegrown, not a pretrained backbone

A large pretrained backbone (e.g. ResNet on ImageNet) "knows" textures from photos of cats and
cars, not vegetation, and expects 3 RGB channels, not 13 reflectance bands — the first layer
would have to be forced, and most of its weights still wouldn't come from this project's data.
With a small, homegrown CNN, every weight was trained on Sentinel-2 imagery of the Chaco: the
question "does the CNN's added complexity pay for itself on this data?" has an honest answer.

In [ ]:
import torch
import torch.nn as nn

class SmallCNN(nn.Module):
    """3 conv blocks (32->64->128 filters) + global pooling + dropout + linear.
    Identical to scripts/10_train_cnn.py -- see the hyperparameter cell below."""

    def __init__(self, in_channels=13, n_classes=3, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


### CNN — hyperparameters, one by one

- **Kernel 3x3, `padding=1`** — 3x3 is the smallest kernel that already looks beyond a single
  pixel; padding keeps the spatial size after each convolution, so pooling (not an unpadded
  convolution) is the only thing responsible for reducing resolution.
- **Filters 32→64→128** — how many distinct patterns each block learns. Doubling at each block is
  the standard convention (each block halves spatial resolution via pooling while doubling
  channel depth, keeping compute balanced). More filters = more capacity for complex patterns,
  but more parameters to overfit with only ~840 training examples — 32/64/128 is deliberately
  modest for that volume.
- **`BatchNorm2d`** — normalizes each channel's activations within a batch, stabilizes training,
  and allows a higher learning rate without diverging.
- **`MaxPool2d(2)` (x2)** — halves spatial resolution each time, keeping the strongest activation
  from each 2x2 block. Goes from 33x33 down to ~8x8 before global pooling.
- **`AdaptiveAvgPool2d(1)` (global average pooling)** — collapses the final feature map into a
  single 128-value vector, averaging each channel spatially. Avoids a giant `Linear` layer after
  flattening the whole map (far fewer parameters) and makes the result independent of exactly
  where in the patch a pattern appeared.
- **`dropout` (0.4)** — randomly zeroes 40% of the neurons right before the final layer, only
  during training (automatically disabled in `model.eval()`). Forces the model not to rely too
  heavily on any single pattern. Typical range 0.3-0.5 for small datasets — raising it a lot
  (e.g. 0.7) can leave the model without enough signal (underfitting); lowering it to 0 removes
  this regularization entirely.
- **`batch_size` (32)** — how many patches are processed together before updating the weights.
  Larger = more stable gradient but fewer updates per epoch and more memory; smaller = noisier
  gradient, sometimes helps generalization with little data. 32 gives ~26 batches per epoch with
  the ~840 training patches.
- **`learning_rate` (0.001)** — step size on each weight update (Adam optimizer). Too high → the
  loss diverges or oscillates without converging; too low → training is very slow or gets stuck
  in a poor minimum within the epoch budget. 0.001 is the default recommended in the original
  Adam paper (Kingma & Ba, 2014).
- **`max_epochs` (100) + `early_stop_patience` (10)** — `max_epochs` is just a safety ceiling; the
  real rule is early stopping: if validation loss doesn't improve for 10 straight epochs, it
  stops and restores the best saved checkpoint. Lets the data itself decide how long to train,
  instead of guessing a fixed number, and avoids overfitting by memorizing the training set past
  the point where it stops helping on validation.
- **`seed_everything`** — sets numpy, `random`, and torch together. Notable: a code review found
  that the seed originally only controlled PyTorch, not the numpy RNG that drives data
  augmentation (rotation/flipping) — so the same seed didn't reproduce the same run until this
  was fixed.

In [ ]:
import random
from torch.utils.data import DataLoader

BATCH_SIZE = 32          # change this for the live demo
LEARNING_RATE = 0.001    # change this for the live demo
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 10
DROPOUT = 0.4            # change this for the live demo

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def run_epoch(model, loader, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.set_grad_enabled(train):
        for patches, labels in loader:
            patches, labels = patches.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(patches)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            n += len(labels)
    return total_loss / n, correct / n

seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_loader = DataLoader(dl.PatchDataset("train", augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dl.PatchDataset("val", augment=False), batch_size=BATCH_SIZE)

cnn = SmallCNN(dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(cnn.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, patience_left, best_state = float("inf"), EARLY_STOP_PATIENCE, None
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(cnn, train_loader, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(cnn, val_loader, optimizer, device, train=False)
    for key, value in [("train_loss", train_loss), ("val_loss", val_loss), ("train_acc", train_acc), ("val_acc", val_acc)]:
        history[key].append(value)
    if epoch % 5 == 0:
        print(f"  epoch {epoch}: train_loss={train_loss:.3f} val_loss={val_loss:.3f} train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss, patience_left = val_loss, EARLY_STOP_PATIENCE
        best_state = {k: v.clone() for k, v in cnn.state_dict().items()}
    else:
        patience_left -= 1
        if patience_left == 0:
            print(f"  early stop at epoch {epoch} (best val_loss={best_val_loss:.3f})")
            break

cnn.load_state_dict(best_state)
print("CNN trained.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("epoch")
fig.tight_layout()
plt.show()


In [ ]:
test_loader = DataLoader(dl.PatchDataset("test", augment=False), batch_size=BATCH_SIZE)
cnn.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for patches, labels in test_loader:
        logits = cnn(patches.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels.append(labels.numpy())
cnn_pred = np.concatenate(all_preds)
cnn_true = np.concatenate(all_labels)
cnn_results = classifier_report.evaluate_predictions(cnn_true, cnn_pred)
classifier_report.print_evaluation(cnn_results, "CNN")


## Step 5 — The real result, and which model is the operational classifier

With the 5 seeds already run (outside this notebook, `scripts/11_compare_classifiers.py`,
persisted to `data/study_area/comparison_results.json`): **RF 0.811 macro F1 (± 0.004), CNN 0.843
(± 0.012)** on the AOI's own test split — the CNN wins there. But Step 2's held-out checks tell a
more complete story: across the AOI's test split plus the 3 independent held-out regions, the CNN
wins or ties in 3 of the 4 contexts (this AOI, Bahía Negra, Pozo Colorado) and loses only in
Filadelfia — by a wider margin than its own AOI win. The single-seed run above should land close
to these 5-seed averages.

**Decision (closed 2026-09-13, `phase1-classifier` task 4.4): the CNN is the operational
classifier**, on the strength of that broader record, not just the AOI split above — with a
documented limitation, not a hidden one: see Step 5's continuation below.

In [ ]:
import json

with open(REPO_ROOT / "data/study_area/comparison_results.json") as f:
    comp = json.load(f)

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([comp["rf_scores"], comp["cnn_scores"]])
ax.set_xticks([1, 2], ["Random Forest", "CNN"])
ax.set_ylabel("Macro F1 (test)")
ax.set_title("5 seeds per model — fixed train/val/test split")
ax.axhline(comp["rf_mean"], color="gray", linestyle="--", linewidth=0.8)
plt.show()

print(f"RF:  mean={comp['rf_mean']:.3f}  std={comp['rf_std']:.3f}")
print(f"CNN: mean={comp['cnn_mean']:.3f}  std={comp['cnn_std']:.3f}")


**Why the CNN wins on the AOI, without that being the whole story**: bosque/pasto/cultivo in
the Chaco are mostly distinguished by spectral signature (SWIR and red-edge, top of RF's
importance ranking above), but spatial context (33x33) can still add something — crop rows
visible to the eye in Step 3's grid are a concrete example of real spatial signal in the data.
With the corrected dataset (no 2019/2023 leak), the CNN captures that signal better than RF on
the AOI's own test split.

**But Filadelfia (Step 2) tells the opposite story, by an even wider margin**: RF beats the CNN
there by 0.133 macro F1 (0.829 vs. 0.696), with the CNN degrading specifically on bosque (IoU
0.812 → 0.457). Filadelfia is the only region genuinely outside the training footprint, so this
gap matters: the CNN can be learning something specific to the training geography that doesn't
generalize, while RF's simpler spectral threshold does.

**Why the CNN is still the operational classifier despite that**: adding 2 more held-out regions
(Bahía Negra, Pozo Colorado) and scaling up training volume ruled out a simple data-volume
explanation — the CNN wins or ties in 3 of the 4 evaluated contexts (this AOI, Bahía Negra 0.611
vs. 0.611 essentially tied, Pozo Colorado 0.727 vs. 0.661) and loses only in Filadelfia. The
Filadelfia loss traces to a specific, understood mechanism: the CNN misreads narrow, fragmented
forest strips (e.g. agricultural-colony windbreaks under ~330m wide) because it partly learned
"bosque" as a large-contiguous-canopy spatial pattern — RF doesn't have this failure mode since it
only ever reads the center pixel. The case-study AOI (this project's actual deployment target) is
not a fragmented-colony landscape like Filadelfia's, so this known failure mode is unlikely to be
triggered there, though not impossible. **Documented limitation, carried forward rather than
hidden**: the CNN shouldn't be trusted, without an RF cross-check, for bosque classification in
landscapes with narrow/fragmented forest strips structurally similar to Filadelfia's.

## What's next (outside this notebook)

- **Grad-CAM**: per-prediction visual evidence on the CNN — not yet implemented, its own OpenSpec
  change. Its value doesn't depend on the CNN's accuracy edge, but it now targets the CNN as the
  settled operational classifier, not a still-open question.
- **Temporal change detection (2019 vs. 2023)**: already implemented separately
  (`phase1-temporal-comparison`) — dense classification over the AOI for both dates, diffed into
  a named-transition change map, cross-checked against MapBiomas' own year-over-year transitions.
- **Phase 2**: carbon estimation (pretrained canopy height + GEDI L4A calibration + IPCC
  conversion to CO2e).